# 第 13 章 · DPO 直接偏好优化

> 📄 配套动画网页：`../13_dpo.html`　（先用浏览器打开，把原理动画看一遍再来写代码）

**所属部分**：第三部分 · 训练

DPO 把 RLHF 那一整套复杂流程，化简成一个二分类式的损失函数。只用现成的好坏答案数据就能对齐。

---


## 🎯 本章目标

- 理解 DPO 损失公式的每一项
- 掌握用 `log_softmax` + `gather` 取特定 token 的 log 概率
- 掌握用 loss_mask 把逐 token logprob 汇总成整句 logprob
- 能实现完整的 `dpo_loss`

---


## 🖼 官方配图

![DPO 用的是偏好数据集 dpo.jsonl（chosen / rejected 成对出现），就在这张数据构成图里](../images/dataset.jpg)

*DPO 用的是偏好数据集 dpo.jsonl（chosen / rejected 成对出现），就在这张数据构成图里*

---


## 📖 背景：一个损失函数搞定偏好对齐

```python
ratio_w = logπ(y_w|x) − logπ_ref(y_w|x)     # 好答案相对参考模型的提升
ratio_l = logπ(y_l|x) − logπ_ref(y_l|x)     # 坏答案的提升

loss = −log σ( β · (ratio_w − ratio_l) )
```

直观理解：**让「好答案」相对参考模型被抬得更高，让「坏答案」被压得更低。**

`π_ref` 是冻结的 SFT 模型。减掉它的 log 概率，相当于只奖励「比原来更好的变化」，
防止模型为了刷分而偏离语言本身。

---



# 第一部分 · 基础函数演示

> 这一部分**可以直接运行**。先把本章要用到的 Python / PyTorch 函数和方法摸熟，后面自己写的时候才不会卡在 API 上。


### 13.1 取特定位置、特定 token 的 log 概率

`log_softmax` + `gather` 是标准组合。

In [ ]:
import torch
import torch.nn.functional as F

logits = torch.randn(2, 5, 10)              # [B, L, vocab]
labels = torch.randint(0, 10, (2, 5))       # 每个位置的目标 token

logp = F.log_softmax(logits, dim=-1)                    # [B, L, vocab]
picked = logp.gather(-1, labels.unsqueeze(-1)).squeeze(-1)   # [B, L]
print("logits :", logits.shape)
print("logp   :", logp.shape)
print("picked :", picked.shape, " <- 每个位置目标 token 的 log 概率")

# 验证：手工取出第一个位置
print("手工验证:", logp[0, 0, labels[0, 0]].item(), "==", picked[0, 0].item())

### 13.2 用 mask 汇总整句的 log 概率

回答只占序列的一部分，要用 mask 把非回答位置去掉。

In [ ]:
mask = torch.tensor([[0, 0, 1, 1, 1], [0, 0, 0, 1, 1]])      # 1 = 属于回答
seq_logp = (picked * mask).sum(dim=-1) / mask.sum(dim=-1).clamp(min=1)
print("每条回答的平均 log 概率:", seq_logp.tolist())

### 13.3 `F.logsigmoid` 与 sigmoid

In [ ]:
x = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])
print("sigmoid    :", torch.sigmoid(x).tolist())
print("logsigmoid :", F.logsigmoid(x).tolist())
print("-log(sigmoid(x)):", (-F.logsigmoid(x)).tolist())

### 13.4 手推一遍 DPO 损失

In [ ]:
beta = 0.1
# 策略模型的 log 概率
logp_w, logp_l = -1.5, -2.8
# 参考模型的 log 概率
ref_w, ref_l = -3.4, -3.2

ratio_w = logp_w - ref_w
ratio_l = logp_l - ref_l
margin = beta * (ratio_w - ratio_l)
loss = -F.logsigmoid(torch.tensor(margin))

print(f"ratio_w = {ratio_w:.3f}   ratio_l = {ratio_l:.3f}")
print(f"margin  = {margin:.3f}")
print(f"loss    = {loss.item():.4f}")
print(f"隐式偏好准确率 σ(margin) = {torch.sigmoid(torch.tensor(margin)).item():.4f}")

# 第二部分 · 动手实现（只给提示，留白给你）

> 下面每一节都**没有答案**。请根据文字提示 + 第一部分演示过的函数，自己从零写出来。
> 写不出来时，先回去看第一部分的演示，再对照网页里的动画想清楚数据形状。

现在实现完整的 DPO 损失。

> 💡 每个留白代码块后面都跟了一个**测试块**。先写代码，再运行测试块。


In [ ]:
# ════════════════════ 自测工具 ════════════════════
# 每个留白代码块后面都跟一个测试块。先在上面写代码，再运行测试块。
# 没写之前运行也不会报错，只会提示「未实现」。
import torch
import torch.nn.functional as F

_PASS, _FAIL, _SKIP = [], [], []

def check(name, fn):
    """运行一个测试。捕获异常并打印结果，不会中断后面的测试。"""
    try:
        fn()
        print(f"✅ 通过   {name}")
        _PASS.append(name)
    except NameError as e:
        print(f"⏸  未实现 {name} —— {e}")
        _SKIP.append(name)
    except AssertionError as e:
        print(f"❌ 失败   {name} —— {e}")
        _FAIL.append(name)
    except Exception as e:
        print(f"❌ 出错   {name} —— {type(e).__name__}: {e}")
        _FAIL.append(name)

def close(a, b, tol=1e-4, msg=""):
    """断言两个张量 / 数字足够接近"""
    a, b = torch.as_tensor(a).float().reshape(-1), torch.as_tensor(b).float().reshape(-1)
    assert a.numel() == b.numel(), f"元素个数不一致 {a.numel()} vs {b.numel()} {msg}"
    assert torch.allclose(a, b, atol=tol), f"数值不接近（容差 {tol}）{msg}"

def same(a, b, msg=""):
    """断言两个张量完全相同"""
    a, b = torch.as_tensor(a), torch.as_tensor(b)
    assert a.shape == b.shape, f"形状不一致 {tuple(a.shape)} vs {tuple(b.shape)} {msg}"
    assert torch.equal(a, b), f"内容不相同 {msg}"

def summary():
    print()
    print(f"通过 {len(_PASS)} · 失败 {len(_FAIL)} · 未实现 {len(_SKIP)}")
    if _FAIL: print("需要修的：", "、".join(_FAIL))
    if _SKIP: print("还没写的：", "、".join(_SKIP))
    if _PASS and not _FAIL and not _SKIP:
        print("本章全部通过 🎉")

print("自测工具已就绪 —— 每写完一个留白，就运行它下面的测试块。")

### 13.5 实现 `compute_logprob`

**要求**：给定 `logits [B, L, V]`、`labels [B, L]`、`mask [B, L]`，
返回每条序列的 log 概率（标量，形状 `[B]`）。

步骤：
1. `logp = F.log_softmax(logits, dim=-1)`
2. `picked = logp.gather(-1, labels.unsqueeze(-1)).squeeze(-1)`
3. `(picked * mask).sum(-1) / mask.sum(-1).clamp(min=1)`

**注意**：labels 里为 -100 的位置要先用 mask 归零，否则 `gather` 会越界。

In [ ]:
# ======== 你的实现：compute_logprob ========
#
# 【要求】
#   - 签名：compute_logprob(logits, labels, mask)
#   - logits: [B, L, V]，labels: [B, L]，mask: [B, L]（1=计入）
#   - 先把 labels 中 -100 的位置替换成 0（避免 gather 越界）
#   - logp = F.log_softmax(logits, -1)
#   - picked = logp.gather(-1, labels.unsqueeze(-1)).squeeze(-1)
#   - 返回 (picked * mask).sum(-1) / mask.sum(-1).clamp(min=1)
#
# 【可用到的函数 / 方法】
#   - `F.log_softmax`
#   - `Tensor.gather`、`Tensor.unsqueeze`、`Tensor.squeeze`
#   - `Tensor.clamp`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：compute_logprob ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    torch.manual_seed(0)
    logits = torch.randn(2, 5, 10)
    labels = torch.randint(0, 10, (2, 5))
    mask = torch.tensor([[0, 0, 1, 1, 1], [0, 0, 0, 1, 1]])

    out = compute_logprob(logits, labels, mask)
    assert tuple(out.shape) == (2,), f"应返回形状 (2,) 的每条序列 log 概率，得到 {tuple(out.shape)}"
    logp = F.log_softmax(logits, -1).gather(-1, labels.unsqueeze(-1)).squeeze(-1)
    ref = (logp * mask).sum(-1) / mask.sum(-1).clamp(min=1)
    close(out, ref, tol=1e-5, msg="与 (logp * mask).sum(-1) / mask.sum(-1) 不一致")

    labels2 = labels.clone(); labels2[:, 0] = -100
    compute_logprob(logits, labels2, mask)   # 不应抛异常

check("compute_logprob 形状 / 数值 / 容错 -100", _t)

### 13.6 实现 `dpo_loss`

**要求**：输入策略模型和参考模型对 chosen / rejected 的 log 概率，返回 DPO 损失。

```
ratio_w = logp_w - ref_logp_w
ratio_l = logp_l - ref_logp_l
loss = -F.logsigmoid(beta * (ratio_w - ratio_l)).mean()
```

**验证**：如果策略模型与参考模型完全一致，loss 应该等于 `log(2) ≈ 0.693`。

In [ ]:
# ======== 你的实现：dpo_loss ========
#
# 【要求】
#   - 签名：dpo_loss(logp_w, logp_l, ref_logp_w, ref_logp_l, beta=0.1)
#   - 四个输入都是形状 [B] 的张量
#   - ratio_w = logp_w - ref_logp_w；ratio_l = logp_l - ref_logp_l
#   - margin = beta * (ratio_w - ratio_l)
#   - return -F.logsigmoid(margin).mean()
#   - （可选）同时返回 chosen/rejected 的隐式奖励供观察
#
# 【可用到的函数 / 方法】
#   - `F.logsigmoid`
#   - `Tensor.mean`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：dpo_loss ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    w, l = torch.tensor([-1.5]), torch.tensor([-2.8])
    rw, rl = torch.tensor([-3.4]), torch.tensor([-3.2])

    out = dpo_loss(w, l, rw, rl, beta=0.1)
    loss = out[0] if isinstance(out, tuple) else out
    ref = -F.logsigmoid(0.1 * ((w - rw) - (l - rl))).mean()
    close(loss, ref, tol=1e-5, msg="与 -log σ(β·(Δw − Δl)) 不一致")

    out2 = dpo_loss(rw, rl, rw, rl, beta=0.1)
    loss2 = out2[0] if isinstance(out2, tuple) else out2
    close(loss2.item(), float(torch.log(torch.tensor(2.0))), tol=1e-5,
          msg="策略模型与参考模型完全一致时 margin=0，loss 应为 log(2) ≈ 0.693")

check("dpo_loss 数值 / 策略=参考时等于 log2", _t)

In [ ]:
summary()

# 第三部分 · 自测与思考

- [ ] 能解释为什么减去参考模型的 log 概率
- [ ] β 变大时损失和梯度会怎样变化？
- [ ] 策略=参考时 loss 是多少？验证了吗？
- [ ] `gather` 之前为什么要把 -100 替换掉？

**思考题**：DPO 只能利用静态的偏好数据。如果模型在训练中生成了一条比数据里更好的回答，DPO 能利用它吗？
